In [82]:
from ete3 import Tree
from Bio import SeqIO
import os
import historydag
from dpvtex.perfect_phylogenies.perturb_phylogeny import (
    sankoff_for_missing_sequences,
)

In [83]:
# Define filepaths to lists of trees and corresponding fastas
test_tree_files = ["influenzaC_fluC_M/influenzaC_fluC_M_log.trees", "influenzaC_fluC_NS/influenzaC_fluC_NS_log.trees", "influenzaC_fluC_PB2/influenzaC_fluC_PB2_log.trees"]
test_tree_files = [f"../data/viral_for_tree_search/{x}" for x in test_tree_files]
test_fastas = [i.replace("_log.trees", ".fasta") for i in test_tree_files]
# dag_files = ["/".join(f.split("/")[:-1]) + "/larch-output.pb" for f in test_tree_files]

# use original dag files from Mary
dag_file_base = "/fh/fast/matsen_e/mbarker/larch/viral-datasets/rerun-larch/"
dag_files = [dag_file_base + f.split("/")[-1].replace("_log.trees", "") + "/larch_OUTPUT_DAG_trimmed.pb" for f in test_tree_files]

In [84]:
test_tree_files.append("../data/viral_for_tree_search/influenzaC_fluC_M/influenzaC_fluC_M_log_new.trees")
dag_files.append(dag_files[0])
test_fastas.append(test_fastas[0])

In [85]:
# Read trees and fastas
parsimony_scores = {} # For each test_tree_file save list of parsimony scores of trees in test_tree_files
for i in range(len(test_tree_files)):
    # Load Trees
    with open(test_tree_files[i], "r") as f:
        trees = f.readlines()
    trees = [Tree(t.strip()) for t in trees]
    # load fasta
    sequences = {}
    for record in SeqIO.parse(test_fastas[i], "fasta"):
        sequences[record.id] = str(record.seq.upper())
    
    # Everything DAG
    # check DAG parsimony score
    dag_file = dag_files[i]
    dag = historydag.mutation_annotated_dag.load_MAD_protobuf_file(
            dag_file, compact_genomes=True
        )
    dag = historydag.sequence_dag.SequenceHistoryDag.from_history_dag(dag)
    print("DAG parsimony score:", dag.hamming_parsimony_count())
    dag_sequences = {}
    for leaf in dag.get_leaves():
        dag_sequences[leaf.label.node_id] = leaf.label.sequence
        if leaf.label.sequence != sequences[leaf.label.node_id]:
            print(f"Warning: Sequence mismatch for leaf {leaf.label.node_id}")
            print(f"  DAG sequence: {leaf.label.sequence}")
            print(f"  Fasta sequence: {sequences[leaf.label.node_id]}")
    print("num_leaves DAG:", len(dag_sequences), len(list(dag.get_leaves())))

    # Add sequences to matching leaves
    parsimony_scores[test_tree_files[i]] = []
    rf_distances_to_dag = []
    for tree in trees:
        # print("num_leaves tree:", len(tree))
        for leaf in tree.get_leaves():
            leaf_name = leaf.name
            if leaf_name in sequences:
                # Add sequence as a feature to the node
                leaf.add_features(sequence=sequences[leaf_name])
            else:
                print(f"Warning: No sequence found for leaf {leaf_name}")
                
        # Compute parsimony score
        sankoff_for_missing_sequences(tree)
        p_score = historydag.parsimony.parsimony_score(tree)
        parsimony_scores[test_tree_files[i]].append(p_score)
    print(len(tree))
    print(len(sequences))
    print(len(set(sequences.values())))
    print(len(set(sequences.keys())))
        # rf_distances_to_dag.append(dag.optimal_rf_distance(historydag.from_tree(tree, ['sequence'])))
    print("RF distances to DAG:")
    print(rf_distances_to_dag)
    best_tree_parsimony_score = min(parsimony_scores[test_tree_files[i]])
    print("Best tree parsimony score from pratchet:", best_tree_parsimony_score)

    if dag_sequences == sequences:
        print("DAG sequences match fasta sequences")
    else:
        print("DAG sequences do not match fasta sequences")

print(parsimony_scores)

DAG parsimony score: Counter({537: 27940744218734729625600000})
num_leaves DAG: 215 215
169
215
198
215
RF distances to DAG:
[]
Best tree parsimony score from pratchet: 471
DAG sequences match fasta sequences
DAG parsimony score: Counter({408: 19926044549774972928})
num_leaves DAG: 165 165
133
166
154
166
RF distances to DAG:
[]
Best tree parsimony score from pratchet: 373
DAG sequences do not match fasta sequences
DAG parsimony score: Counter({859: 1096131117422016000})
num_leaves DAG: 135 135
117
136
135
136
RF distances to DAG:
[]
Best tree parsimony score from pratchet: 844
DAG sequences do not match fasta sequences
DAG parsimony score: Counter({537: 27940744218734729625600000})
num_leaves DAG: 215 215
169
215
198
215
RF distances to DAG:
[]
Best tree parsimony score from pratchet: 473
DAG sequences match fasta sequences
{'../data/viral_for_tree_search/influenzaC_fluC_M/influenzaC_fluC_M_log.trees': [1816, 1713, 1659, 1641, 1630, 1629, 1629, 525, 522, 522, 476, 476, 473, 471], '../

In [86]:
# Check parsimony score in dags

for dag_file in dag_files:
    dag = historydag.mutation_annotated_dag.load_MAD_protobuf_file(
            dag_file, compact_genomes=True
        )
    print(dag.summary())
    dag_sequences = {}
    for leaf in dag.get_leaves():
        dag_sequences[leaf.name] = leaf.sequence

<class 'historydag.mutation_annotated_dag.AmbiguousLeafCGHistoryDag'>
Label fields:	('compact_genome', 'node_id')
Nodes:	797
Edges:	1871
Histories:	27940744218734729625600000
Unique leaves in DAG:	215
Average number of parents of internal nodes:	1.8041237113402062

In histories in the DAG:
Leaf node count range: 215 to 215
Total node count range: 325 to 428
Average pairwise RF distance:	56.686838409412175
Parsimony score range 493 to 493
None


AttributeError: 'HistoryDagNode' object has no attribute 'sequence'